In [3]:
from src.chart_transport.base import ChartTransportConfig
from src.chart_transport.constraint import (
    LagrangianConstraintConfig,
    ManifoldConstraintConfig,
)
from src.chart_transport.field import UniformVelocityMatchingSchedule
from src.chart_transport.loss import ChartTransportLossConfig, TransportLossConfig
from src.chart_transport.model import ChartTransportModelConfig
from src.common.training import TrainingConfig
from src.data.gaussian_mixture.data import MultimodalGaussianDataConfig
from src.model.mlp import StackedResidualMLPConfig
from src.model.time_conditioning import TimeConditioningConfig
from src.priors.anchored import AnchoredGaussianScaleMixturePriorConfig

In [7]:
num_modes = 4
mode_std = 0.1
offset = 1.0
ambient_dimension = 8
latent_dimension = 8
prior_precision = 4.0

data_config = MultimodalGaussianDataConfig.initialize(
    num_modes=num_modes,
    mode_std=mode_std,
    offset=offset,
    ambient_dimension=ambient_dimension,
)

prior_config = AnchoredGaussianScaleMixturePriorConfig.initialize(
    data_shape=[latent_dimension],
    precision=prior_precision,
)

In [8]:
constraint_method = LagrangianConstraintConfig(
    data_constraint_budget=0.01,
    prior_constraint_budget=0.01,
    dual_variable_lr=1e-3,
)

constraint_config = ManifoldConstraintConfig(
    huber_delta=0.25,
    constraint_method=constraint_method,
)

kl_weight_schedule = UniformVelocityMatchingSchedule()

transport_config = TransportLossConfig(
    kl_weight_schedule=kl_weight_schedule,
    transport_step_size=0.1,
    num_time_samples=8,
    antipodal_estimate=True,
    decoder_transport_weight=1.0,
    encoder_transport_weight=1.0,
)

loss_config = ChartTransportLossConfig(
    constraint_config=constraint_config,
    transport_config=transport_config,
    critic_loss_weight=1.0,
    update_chart_every_n_critic_steps=4,
)

display(loss_config.visualize())

In [6]:
hidden_dimension = 128
time_embedding_dimension = 64

encoder_config = StackedResidualMLPConfig.initialize(
    layer_dims=[ambient_dimension, hidden_dimension, hidden_dimension, latent_dimension],
)

decoder_config = StackedResidualMLPConfig.initialize(
    layer_dims=[latent_dimension, hidden_dimension, hidden_dimension, ambient_dimension],
)

critic_time_conditioning_config = TimeConditioningConfig(
    min_t_lambda=1e-3,
    max_t_lambda=1.0,
    sinusoidal_dim=time_embedding_dimension,
    hidden_dim=hidden_dimension,
    output_dim=time_embedding_dimension,
)

critic_config = StackedResidualMLPConfig.initialize(
    layer_dims=[latent_dimension, hidden_dimension, hidden_dimension, latent_dimension],
    time_conditioning_config=critic_time_conditioning_config,
)

architecture_config = ChartTransportModelConfig(
    encoder=encoder_config,
    decoder=decoder_config,
    critic=critic_config,
    chart_lr=1e-3,
    critic_lr=1e-3,
)

chart_transport_config = ChartTransportConfig(
    data_config=data_config,
    prior_config=prior_config,
    loss_config=loss_config,
    architecture_config=architecture_config,
)

training_config = TrainingConfig(
    train_batch_size=256,
    eval_batch_size=1024,
)

display(architecture_config.visualize())
display(chart_transport_config.visualize())
display(training_config.visualize())